In [2]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from configs import config

# Project Root
PROJECT_ROOT = config.PROJECT_ROOT

# Data Paths
RAW_DATA_PATH = config.RAW_DATA_FILE
PROCESSED_DATA_PATH = config.PROCESSED_DATA_FILE

# Artifact Directory
ARTIFACT_DIR = config.ARTIFACT_DIR
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing processed data file: {PROCESSED_DATA_PATH}")

FileNotFoundError: Missing processed data file: C:\Users\silsw\OneDrive\Desktop\PV-forecasting\data\processed\Processed.csv

In [5]:
df = pd.read_csv(
    PROCESSED_DATA_PATH,            
    parse_dates=["timestamp"]
)

print(f"Dataset Shape : {df.shape}")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\silsw\\OneDrive\\Desktop\\PV-forecasting\\data\\processed\\Processed.csv'

In [ ]:
TARGET_COLUMN = "Active_Power"

FEATURE_COLUMNS = [
    "Weather_Temperature_Celsius",
    "Weather_Relative_Humidity",
    "Global_Horizontal_Radiation",
    "Diffuse_Horizontal_Radiation",
    "Radiation_Global_Tilted",
    "Radiation_Diffuse_Tilted"
]

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

Feature Shape : (1065289, 6)
Target Shape  : (1065289,)


In [ ]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

n_samples = len(df)

train_end = int(TRAIN_RATIO * n_samples)
val_end = train_end + int(VAL_RATIO * n_samples)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

print(f"Train Samples      : {len(X_train)}")
print(f"Validation Samples : {len(X_val)}")
print(f"Test Samples       : {len(X_test)}")

Train Samples      : 745702
Validation Samples : 159793
Test Samples       : 159794


In [ ]:
feature_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)

X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

In [ ]:
target_scaler = StandardScaler()

y_train_scaled = target_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1))

y_val_scaled = target_scaler.transform(y_val.to_numpy().reshape(-1, 1))
y_test_scaled = target_scaler.transform(y_test.to_numpy().reshape(-1, 1))

In [ ]:
LOOKBACK = 24          # Previous 2 hours
HORIZONS = [3, 12]     # 15 min and 60 min
STRIDE = 1

In [ ]:
def create_sequences(features, target, lookback, horizon, stride=1):
    
    X_seq = []
    y_seq = []

    last_start = len(features) - lookback - horizon + 1

    for start in range(0, last_start, stride):

        end = start + lookback

        X_seq.append(features[start:end])

        y_seq.append(target[end:end + horizon, 0])

    X_seq = torch.tensor(np.array(X_seq), dtype=torch.float32)
    y_seq = torch.tensor(np.array(y_seq), dtype=torch.float32)

    return X_seq, y_seq

In [ ]:
datasets = {}

for horizon in HORIZONS:

    X_train_seq, y_train_seq = create_sequences(
        X_train_scaled,
        y_train_scaled,
        LOOKBACK,
        horizon,
        STRIDE,
    )

    X_val_seq, y_val_seq = create_sequences(
        X_val_scaled,
        y_val_scaled,
        LOOKBACK,
        horizon,
        STRIDE,
    )

    X_test_seq, y_test_seq = create_sequences(
        X_test_scaled,
        y_test_scaled,
        LOOKBACK,
        horizon,
        STRIDE,
    )

    datasets[horizon] = {
        "train": (X_train_seq, y_train_seq),
        "val": (X_val_seq, y_val_seq),
        "test": (X_test_seq, y_test_seq),
    }

    print(f"\n===== {horizon * 5} Minute Forecast =====")
    print("Train :", X_train_seq.shape, y_train_seq.shape)
    print("Val   :", X_val_seq.shape, y_val_seq.shape)
    print("Test  :", X_test_seq.shape, y_test_seq.shape)


===== 15 Minute Forecast =====
Train : torch.Size([745676, 24, 6]) torch.Size([745676, 1])
Val   : torch.Size([159767, 24, 6]) torch.Size([159767, 1])
Test  : torch.Size([159768, 24, 6]) torch.Size([159768, 1])

===== 60 Minute Forecast =====
Train : torch.Size([745667, 24, 6]) torch.Size([745667, 1])
Val   : torch.Size([159758, 24, 6]) torch.Size([159758, 1])
Test  : torch.Size([159759, 24, 6]) torch.Size([159759, 1])


In [ ]:
for horizon in HORIZONS:

    torch.save(
        {
            "X": datasets[horizon]["train"][0],
            "y": datasets[horizon]["train"][1],
        },
        ARTIFACT_DIR / f"train_{horizon * 5}.pt",
    )

    torch.save(
        {
            "X": datasets[horizon]["val"][0],
            "y": datasets[horizon]["val"][1],
        },
        ARTIFACT_DIR / f"val_{horizon * 5}.pt",
    )

    torch.save(
        {
            "X": datasets[horizon]["test"][0],
            "y": datasets[horizon]["test"][1],
        },
        ARTIFACT_DIR / f"test_{horizon * 5}.pt",
    )

In [ ]:
with open(ARTIFACT_DIR / "feature_scaler.pkl", "wb") as f:
    pickle.dump(feature_scaler, f)

with open(ARTIFACT_DIR / "target_scaler.pkl", "wb") as f:
    pickle.dump(target_scaler, f)

In [ ]:
config = {
    "lookback": LOOKBACK,
    "horizons": HORIZONS,
    "stride": STRIDE,
    "features": FEATURE_COLUMNS,
    "target": TARGET_COLUMN,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "feature_scaler": "StandardScaler",
    "target_scaler": "StandardScaler",
}

with open(ARTIFACT_DIR / "preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=4)

In [ ]:
print("Artifacts Saved Successfully\n")

for file in sorted(ARTIFACT_DIR.iterdir()):
    print(file.name)

Artifacts Saved Successfully

feature_scaler.pkl
preprocessing_config.json
target_scaler.pkl
test_15.pt
test_60.pt
train_15.pt
train_60.pt
val_15.pt
val_60.pt
